<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ChemNLP_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Example to run [ChemNLP](https://github.com/usnistgov/chemnlp)

ChemNLP is a Natural Language Processing based Library for Materials Chemistry Text Data.

Rference: https://arxiv.org/abs/2209.08203

# Table of contents

1. Install ChemNLP
2. Parsing chemical formula from text
3. Dataset download and visualization
4. Clustering using t-SNE
5. Classification using Logistic regression
6. Text generation using HuggingFace interface
7. JARVIS-Leaderboard entry

Author: Kamal Choudhary (kamal.choudhary@nist.gov)

# 1. Installation

In [ ]:
%%time
!wget -O mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py39_4.9.2-Linux-x86_64.sh
!chmod +x mini.sh
!bash ./mini.sh -b -f -p /usr/local
!conda install -q -y jupyter
!conda install -q -y google-colab -c conda-forge
!python -m ipykernel install --name "py39" --user

In [ ]:
!easy_install pip

In [ ]:
%%time
!pip  install -U chemnlp

In [ ]:
!pip install -q scikit-learn==1.1.2 seaborn jarvis-tools

In [ ]:
!cde data download

In [ ]:
import os
!pwd
os.chdir('/content')
# Clone chemnlp repo to get example folder
if not os.path.exists('chemnlp'):
  !git clone https://github.com/usnistgov/chemnlp.git
os.chdir('chemnlp')



# 2. Parsing chemical formula from text

In [ ]:
!python chemnlp/utils/run_chemnlp.py --file_path="chemnlp/tests/XYZ"

# 3. Dataset on FigShare: https://doi.org/10.6084/m9.figshare.22351717

In [ ]:
!wget https://figshare.com/ndownloader/files/39768544 -O cond_mat.zip
!unzip cond_mat.zip
!wget https://figshare.com/ndownloader/files/39769168 -O pubchem.zip
!unzip pubchem.zip
!ls

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline
cond_mat=pd.read_csv('cond_mat.csv')
ax = cond_mat['categories'].value_counts().plot(kind='barh',color='#2ab0ff')
ax.invert_yaxis()

In [ ]:
pubchem=pd.read_csv('pubchem.csv')
ax = pubchem['label_name'].value_counts().plot(kind='barh',color='#2ab0ff')
ax.invert_yaxis()

# 4. Clustering using t-SNE

In [ ]:
%%time
from chemnlp.clustering.tsne import tsne
cond_mat=pd.read_csv('cond_mat.csv')[0:1000]
tsne(df=cond_mat,category_key='categories',text='title')

In [ ]:
%%time
pubchem=pd.read_csv('pubchem.csv')[0:1000]
tsne(df=pubchem,category_key='label_name',text='title')

# 4. Classification using Logistic regression

In [ ]:
%%time
#!python chemnlp/classification/scikit_class.py --csv_path chemnlp/sample_data/cond_mat_small.csv
!python chemnlp/classification/scikit_class.py --csv_path pubchem.csv --key_column label_name --value_column title

In [ ]:
pred_test = pd.read_csv('pred_test.csv')

In [ ]:
pred_test

In [ ]:
key='label_name'
pubchem["category_id"] = pubchem[key].factorize()[0]
category_id_df = (
    pubchem[[key, "category_id"]].sort_values("category_id")
)

category_to_id = dict(category_id_df.values)
keys = list(category_to_id.keys())
print("category_to_id", category_to_id)

Plot Confusion materix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
y_pred = pred_test['prediction']
y_test = pred_test['target']
plt.rcParams.update({"font.size": 12})
conf_mat = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8,8))
sns.heatmap(
    conf_mat / conf_mat.sum(axis=1)[:, np.newaxis],
    annot=True,
    # fmt="d",
    fmt=".1%",
    cbar=False,
    square=True,
    cmap=sns.diverging_palette(20, 220, n=200),
    xticklabels=keys,
    yticklabels=keys,
)
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()

# [JARVIS-Leaderboard](https://pages.nist.gov/jarvis_leaderboard/) entry
# See also: https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ChemNLP_TitleToAbstract.ipynb

In [ ]:
pip freeze